## Case When formatting test

### Goals

Answer question from `orbital` team: 

> Does "unnesting" queries improve performance for trees? If only sometimes, where/when?

Research design:

- Simulate SQL code for medium-size random forest:
  + Max Depth: 4 (2**4 terminal nodes)
  + Trees: 100
- Test three different varieties for performance:
  + Original nested CASE WHEN
  + "Linearized" CASE WHEN (pulling all conditions to top level)
  + "Smart" Linearized -- same as above but with refundant conditions removed
- Test on two different data distributions:
  + Uniformly distributed (all nodes have same chance of terminating)
  + Skewed distribution (to simulate if we re-rodered query to take advantage of CASE WHEN early term)


### Set Up -- Tree Code

In [23]:
import duckdb
import polars as pl
import polars.selectors as cs
import sqlglot
import pandas as pd
import numpy as np

In [24]:
# nested case when version
d = []
for i in np.arange(8):
    d += [ f"case when d{i+1} > 0.5 then {i+1} else 0 end"]

c = []
for i in np.arange(4):
    sql_temp = f"case when c{i+1} > 0.5 then {d[2*i]} else {d[2*i+1]} end"
    c += [sql_temp]

b = []
for i in np.arange(2):
    sql_temp = f"case when b{i+1} > 0.5 then {c[2*i]} else {c[2*i+1]} end"
    b += [sql_temp]

sql_nest = f"case when a > 0.5 then {b[0]} else {b[1]} end"

In [25]:
# linearize case when version 
# using strict versus equalities versus inclusion / exclusion just so my code lines up nice =) 
# it's a timing exercise so it really doesn't matter
sql_line = '''
case
-- a > 0.5
when a > 0.5 and b1 > 0.5 and c1 > 0.5 and d1 > 0.5 then 1
when a > 0.5 and b1 > 0.5 and c1 > 0.5 and d1 < 0.5 then 0 
when a > 0.5 and b1 > 0.5 and c1 < 0.5 and d2 > 0.5 then 2
when a > 0.5 and b1 > 0.5 and c1 < 0.5 and d2 < 0.5 then 0
when a > 0.5 and b1 < 0.5 and c2 > 0.5 and d3 > 0.5 then 3
when a > 0.5 and b1 < 0.5 and c2 > 0.5 and d3 < 0.5 then 0 
when a > 0.5 and b1 < 0.5 and c2 < 0.5 and d4 > 0.5 then 4
when a > 0.5 and b1 < 0.5 and c2 < 0.5 and d4 < 0.5 then 0
-- a <= 0.5
when a < 0.5 and b2 > 0.5 and c3 > 0.5 and d5 > 0.5 then 5
when a < 0.5 and b2 > 0.5 and c3 > 0.5 and d5 < 0.5 then 0 
when a < 0.5 and b2 > 0.5 and c3 < 0.5 and d6 > 0.5 then 6
when a < 0.5 and b2 > 0.5 and c3 < 0.5 and d6 < 0.5 then 0
when a < 0.5 and b2 < 0.5 and c4 > 0.5 and d7 > 0.5 then 7
when a < 0.5 and b2 < 0.5 and c4 > 0.5 and d7 < 0.5 then 0 
when a < 0.5 and b2 < 0.5 and c4 < 0.5 and d8 > 0.5 then 8
when a < 0.5 and b2 < 0.5 and c4 < 0.5 and d8 < 0.5 then 0
else null end
'''

In [26]:
# linearized case when version -- pruning redundant conditions
sql_slim = '''
case

when a > 0.5 and b1 > 0.5 and c1 > 0.5 and d1 > 0.5 then 1
when a > 0.5 and b1 > 0.5 and c1 > 0.5              then 0 
when a > 0.5 and b1 > 0.5              and d2 > 0.5 then 2
when a > 0.5 and b1 > 0.5                           then 0
when a > 0.5 and              c2 > 0.5 and d3 > 0.5 then 3
when a > 0.5 and              c2 > 0.5              then 0 
when a > 0.5 and                           d4 > 0.5 then 4
when a > 0.5                                        then 0

when             b2 > 0.5 and c3 > 0.5 and d5 > 0.5 then 5
when             b2 > 0.5 and c3 > 0.5              then 0 
when             b2 > 0.5              and d6 > 0.5 then 6
when             b2 > 0.5                           then 0
when                          c4 > 0.5 and d7 > 0.5 then 7
when                          c4 > 0.5              then 0 
when                                       d8 > 0.5 then 8
when a < 0.5                                        then 0
else null end
'''

### Set Up -- Data

In [27]:
# set up random data matrix
n = 1000000
p = 15
df = pl.DataFrame( np.random.rand(n,p) )
df.columns = ['a','b1','b2','c1','c2','c3','c4','d1','d2','d3','d4','d5','d6','d7','d8']
df.glimpse()

Rows: 1000000
Columns: 15
$ a  <f64> 0.24238669202631946, 0.2735332700758095, 0.937916383491594, 0.1987090242757773, 0.32189422219372377, 0.17232768742250826, 0.4125789975242421, 0.8965242450934352, 0.8228554605609112, 0.4098209320952553
$ b1 <f64> 0.08079309576622207, 0.32901226015181984, 0.07664590281330153, 0.8403629013294024, 0.11476743091207897, 0.1597849982833479, 0.3616981062582001, 0.9546000832681004, 0.6189631421598403, 0.3964744844205862
$ b2 <f64> 0.18096118211192236, 0.30779086437307046, 0.24089611582248205, 0.20568794717949712, 0.6707971623328681, 0.3442679190015665, 0.7246494729915228, 0.6714819335565315, 0.6257524244661282, 0.9707886918606328
$ c1 <f64> 0.4391667664439033, 0.7606632610929583, 0.7187349566168119, 0.023087939614048758, 0.7896138165800813, 0.014775311671888502, 0.6797725618935392, 0.4524504982198647, 0.8819979904143501, 0.8771657498567481
$ c2 <f64> 0.23687218868679893, 0.5546971913866499, 0.4976797581362995, 0.8165634587439295, 0.2993770972959743, 0.865902

In [28]:
# ensure different candidates have same logic
sql_compare = f"""
select
{sql_nest} as out_nest,
{sql_line} as out_line,
{sql_slim} as out_slim,
*
from df
"""
df_out = duckdb.sql(sql_compare).pl()
df_out.filter( 
    (pl.col('out_line') != pl.col('out_slim')) |
    (pl.col('out_nest') != pl.col('out_slim'))
)

out_nest,out_line,out_slim,a,b1,b2,c1,c2,c3,c4,d1,d2,d3,d4,d5,d6,d7,d8
i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64


In [29]:
# base case output frequency
df_out.group_by('out_nest').len()

out_nest,len
i32,u32
5,62936
2,62458
8,62549
6,62248
0,500230
7,62298
1,62474
3,62300
4,62507


In [30]:
# create different skews
df_skew = (
df_out
.with_columns(threshhold = pl.when(pl.col('out_nest') > 0).then(10 - pl.col('out_nest')).otherwise(5))
.filter( pl.col('out_nest').cum_count().over('out_nest') <= pl.col('threshhold')*7000 )
)

print(df_skew.shape[0])

(
df_skew
.group_by('out_nest')
.len()
.sort('out_nest')
.with_columns( p = pl.col('len') / pl.col('len').sum() )
)

342474
342474


out_nest,len,p
i32,u32,f64
0,35000,0.102198
1,62474,0.18242
2,56000,0.163516
3,49000,0.143077
4,42000,0.122637
5,35000,0.102198
6,28000,0.081758
7,21000,0.061319
8,14000,0.040879


In [31]:
# final prep - standardizing data size
df_skew = pl.concat([df_skew]*4).drop( cs.starts_with('out_') )
df_unif = pl.DataFrame( np.random.rand( df_skew.shape[0],p) )
df_unif.columns = ['a','b1','b2','c1','c2','c3','c4','d1','d2','d3','d4','d5','d6','d7','d8']

### Timing

In [32]:
con = duckdb.connect()
con.sql("SET enable_object_cache = false;")

#### Base Case

In this case all nodes are equally likely

In [33]:
qry_nest = f"select {'+'.join([sql_nest]*100)} as pred from df_unif"
qry_line = f"select {'+'.join([sql_line]*100)} as pred from df_unif"
qry_slim = f"select {'+'.join([sql_slim]*100)} as pred from df_unif"
qry_cte1 = f"""
    with trees as (
    select { ','.join([f"{sql_slim} as t{i}" for i in np.arange(100)]) } 
    from df_unif
    ) 
    select {'+'.join([f"t{i}" for i in np.arange(100)])} as pred 
    from trees
    """
qry_cte2 = f"""
    with trees as (
    select { ','.join([f"{sql_nest} as t{i}" for i in np.arange(100)]) } 
    from df_unif
    ) 
    select {'+'.join([f"t{i}" for i in np.arange(100)])} as pred 
    from trees
    """

In [34]:
%%timeit -n 1 -r 50

con.sql(qry_nest).execute()

406 ms ± 18 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
406 ms ± 18 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [35]:
%%timeit -n 1 -r 50

con.sql(qry_line).execute()

1.99 s ± 1.18 s per loop (mean ± std. dev. of 50 runs, 1 loop each)
1.99 s ± 1.18 s per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [36]:
%%timeit -n 1 -r 50

con.sql(qry_slim).execute()

1.74 s ± 822 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
1.74 s ± 822 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [37]:
%%timeit -n 1 -r 50

con.sql(qry_cte1).execute()

The slowest run took 5.56 times longer than the fastest. This could mean that an intermediate result is being cached.
621 ms ± 174 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
The slowest run took 5.56 times longer than the fastest. This could mean that an intermediate result is being cached.
621 ms ± 174 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [38]:
%%timeit -n 1 -r 50

con.sql(qry_cte2).execute()

106 ms ± 5.24 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
106 ms ± 5.24 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


#### Skewed Case

In this case, nodes are skewed, so we can see benefit from the linearized version ordering by node size 

In [39]:
qry_nest = f"select {'+'.join([sql_nest]*100)} as pred from df_skew"
qry_line = f"select {'+'.join([sql_line]*100)} as pred from df_skew"
qry_slim = f"select {'+'.join([sql_slim]*100)} as pred from df_skew"

In [40]:
%%timeit -n 1 -r 50

con.sql(qry_nest).execute()

471 ms ± 32.4 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
471 ms ± 32.4 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [41]:
%%timeit -n 1 -r 50

con.sql(qry_line).execute()

1.7 s ± 38.4 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
1.7 s ± 38.4 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [42]:
%%timeit -n 1 -r 50

con.sql(qry_slim).execute()

1 s ± 33.5 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
1 s ± 33.5 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [43]:
%%timeit -n 1 -r 50

con.sql(qry_cte1).execute()

266 ms ± 10.9 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
266 ms ± 10.9 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [44]:
%%timeit -n 1 -r 50

con.sql(qry_cte2).execute()

104 ms ± 10 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
104 ms ± 10 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


#### Sizes

In [45]:
def kb(s): return len(s.encode('utf-8')) / 1e3

In [46]:
qry_titl = ['Nest', 'Line', 'Slim', 'CTE - Linear', 'CTE - Nest']
qry_lean = [
sqlglot.transpile(qry_nest, write='duckdb', identify=False, pretty=False, indent=0, pad=0)[0],
sqlglot.transpile(qry_line, write='duckdb', identify=False, pretty=False, indent=0, pad=0)[0],
sqlglot.transpile(qry_slim, write='duckdb', identify=False, pretty=False, indent=0, pad=0)[0],
sqlglot.transpile(qry_cte1, write='duckdb', identify=False, pretty=False, indent=0, pad=0)[0],
sqlglot.transpile(qry_cte2, write='duckdb', identify=False, pretty=False, indent=0, pad=0)[0],
]
qry_size = [kb(q) for q in qry_lean]

pl.DataFrame({
    'Query': qry_titl,
    'Size (KB)': qry_size
}).sort('Size (KB)')

Query,Size (KB)
str,f64
"""Nest""",52.825
"""CTE - Nest""",54.038
"""Slim""",56.925
"""CTE - Linear""",58.138
"""Line""",96.525
